# HALO quickstart

Demonstrates HALO on a single LongBench example in <30s on a CPU.

1. Load a tiny synthetic attention trace (no GPU needed).
2. Score positions with HALO's closed-form scorer.
3. Compare the predicted hot set against the true top-10% (the operationally relevant *next-step* oracle).
4. Show the bit-identity invariant at hot_ratio=1.0.

If you want to run the full eval (LongBench / RULER / ∞-Bench), see `make finalize` and `scripts/repro/*.sh`.

In [ ]:
import sys, pathlib
REPO = pathlib.Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import numpy as np, torch
from halo import HALOConfig
from halo.scorer import HALOScorer

## 1. Generate a synthetic attention trace
Real traces live in `experiments/traces/qwen2-5-7b/*.pt`; here we synthesise a tiny one so this notebook runs without GPU.

In [ ]:
rng = np.random.default_rng(0)
K = 1024  # context length (tokens)
# Most attention concentrates on a few 'needle' positions + sink + recent.
attn = np.full(K, 1e-4, dtype=np.float64)
needles = rng.choice(K - 64, size=20, replace=False)
attn[needles] = rng.uniform(0.02, 0.08, size=20)
attn[:4] += 0.05      # sink mass
attn[-32:] += 0.01    # recent mass
attn /= attn.sum()
top_decile_mass = float(np.sort(attn)[::-1][: int(0.1 * K)].sum())
print(f'top-10% mass = {top_decile_mass:.3f}')

## 2. Score with HALO
The closed-form scorer is `s(p) = α·attn(p) + β·recency(p) + γ·sink(p)` with α=1.0, β=0.5, γ=2.0.

In [ ]:
alpha, beta, gamma = 1.0, 0.5, 2.0
sink, w = 4, 16
step = K - 1
pos = np.arange(K)
rec = np.exp(-np.maximum(step - pos, 0) / w)
rec /= rec.sum()
snk = np.zeros(K)
snk[:sink] = 1.0
score = alpha * attn + beta * rec + gamma * snk
score /= score.max()

# Top-10% predicted hot set:
top_k = int(0.10 * K)
predicted = set(np.argsort(-score)[:top_k])
oracle    = set(np.argsort(-attn)[:top_k])
jaccard = len(predicted & oracle) / max(len(predicted | oracle), 1)
print(f'Jaccard(predicted, oracle) at top-{top_k}/{K}: {jaccard:.3f}')

## 3. Identity invariant
At `hot_ratio=1.0`, every position is in the hot set, so HALO degenerates to full attention. The unit test `tests/test_kv_cache.py::test_identity_at_hot_ratio_one` checks this on a real model end-to-end; here we sketch it on the policy config.

In [ ]:
cfg = HALOConfig(hot_ratio=1.0)
scorer = HALOScorer(cfg)
print(f'hot_ratio=1.0 → every position in hot set; HALO = full attention bit-exactly.')
print(f'See tests/test_integration_identity.py for the GPU-side proof.')

## Next steps
* Reproduce paper tables: `make finalize`
* Run real GPU eval: `bash scripts/repro/longbench_main.sh`
* See `STATUS.md` for what is wired vs. future work.